#### LangSmith 기본 예제


##### 1) 라이브러리 설치

In [ ]:
# uv add langsmith

##### 2) 환경 변수 설정 (Groq, LangSmith)

- `.env` 파일에 `GROQ_API_KEY`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` 등록
- `load_dotenv()`가 `.env` 값을 환경 변수로 설정하므로 LangSmith가 별도 코드 없이 읽어서 사용

In [1]:
import os
from dotenv import load_dotenv
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 필수 환경 변수 확인 (값은 출력하지 않고 설정 여부만 확인)
REQUIRED_ENV = ["GROQ_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f".env 파일에 다음 환경 변수가 없습니다: {', '.join(missing)}")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# LANGSMITH_TRACING 이 'true' 여야 트레이스가 LangSmith 로 전송됨
print("LangSmith 트레이싱:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

LangSmith 트레이싱: true
LangSmith 프로젝트: MyLangchainProject


##### LangSmith와 LangChain을 활용한 기본 로깅 예제

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langsmith import traceable

# LangSmith 설정(LANGSMITH_TRACING / API_KEY / PROJECT / ENDPOINT)은
# 위에서 load_dotenv() 로 환경 변수에 이미 로드되어 있으므로 다시 대입할 필요가 없음

# LLM 모델 설정 (Groq 사용: OpenAI 호환 API)
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",  
    temperature=0.7
)

# LangSmith로 실행 추적
@traceable(run_type="chain", name="Simple_Chain")
def ask_question(question: str):

    # 개별 메시지 템플릿 정의
    system_message = SystemMessagePromptTemplate.from_template(
        "당신은 유용한 AI 비서입니다."
    )
    user_message = HumanMessagePromptTemplate.from_template(
        "{question}"
    )

    chat_prompt = ChatPromptTemplate.from_messages([
        system_message,
        user_message,
    ])
    
    messages = chat_prompt.format_messages(question=question)
    response = llm.invoke(messages)

    return response.content

# 테스트 실행
question = "LangGraph와 LangChain의 차이점은 무엇인가요?"
answer = ask_question(question)

# 결과 출력
print("\n🔹 [AI 답변]:")
print(answer)



🔹 [AI 답변]:
## LangChain vs. LangGraph  
*(두 프레임워크 모두 LLM‑기반 애플리케이션을 빠르게 만들기 위한 도구이지만, 설계 철학·구조·주된 활용 시나리오에서 차이가 있습니다.)*

| 구분 | **LangChain** | **LangGraph** |
|------|----------------|---------------|
| **핵심 목표** | LLM을 **프롬프트·체인·에이전트** 로 구성해 **단일 작업** 혹은 **다중 단계 워크플로** 를 빠르게 구현 | LLM을 **노드·엣지 기반 그래프** 로 모델링해 **복잡한 상태 전이·조건부 흐름** 을 명시적으로 표현 |
| **주요 개념** | • **Chain** (Prompt → LLM → Output) <br>• **Agent** (Tool‑calling, Re‑act) <br>• **Memory** (대화 컨텍스트) | • **Node** (작업 단위) <br>• **Edge** (조건·트리거) <br>• **Graph** (전체 상태 머신) |
| **프로그램 모델** | 함수형/객체지향 스타일: `chain.run()`, `agent.run()` 등 순차·루프형 흐름을 코드로 정의 | 선언형 그래프 DSL: `graph.add_node()`, `graph.add_edge(condition)` 로 **상태 전이 그래프** 를 정의하고, 런타임이 자동으로 흐름을 관리 |
| **상태 관리** | 메모리 객체(`ConversationBufferMemory`, `ConversationSummaryMemory` 등) 로 대화 히스토리를 저장. 복잡한 상태 전이는 직접 구현해야 함 | **GraphState** 라는 전역 상태 객체가 자동으로 전달·업데이트. 노드 실행 결과가 다음 노드의 입력이 되며, 조건부 전이는 엣지에 선언된 함수로 처리 |
| **조건·루프** | `while`, `if` 등을 파이썬 코드에 직접 작성. 복잡한 분기·재귀는 코드가 길어짐 | *